In [ ]:
from pathlib import Path
import sys, zipfile, numpy as np, pandas as pd

def find_data_dir():
    local_candidates = [Path.cwd() / "ai_model" / "data" / "single_sensor", Path.cwd().parent / "data" / "single_sensor"]
    for candidate in local_candidates:
        if candidate.is_dir() and list(candidate.glob("*.csv")):
            return candidate
    kaggle_root = Path("/kaggle/input")
    csv_files = list(kaggle_root.rglob("*.csv")) if kaggle_root.is_dir() else []
    if csv_files:
        return csv_files[0].parent
    zip_files = list(kaggle_root.rglob("*.zip")) if kaggle_root.is_dir() else []
    if zip_files:
        output = Path("/kaggle/working/single_sensor_dataset")
        output.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_files[0]) as archive:
            archive.extractall(output)
        extracted_csv = list(output.rglob("*.csv"))
        if extracted_csv:
            return extracted_csv[0].parent
    raise FileNotFoundError("Upload the single_sensor ZIP as a Kaggle Dataset.")

DATA_DIR = find_data_dir()
REPO_ROOT = next((candidate for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working")] if (candidate / "ai_model" / "feature_engineering.py").exists()), Path.cwd())
ROOT = REPO_ROOT
sys.path.insert(0, str(REPO_ROOT))

try:
    from ai_model.feature_engineering import load_recordings, FEATURE_COLUMNS
except ModuleNotFoundError:
    FEATURE_COLUMNS = ["distance_cm", "approach_speed_cm_s"]
    LABELS = {"SAFE": 0, "APPROACHING": 1, "DANGER": 2, "EMERGENCY": 3, "RETREATING": 0}

    def load_recordings(folder):
        frames = []
        for csv_path in sorted(Path(folder).glob("*.csv")):
            frame = pd.read_csv(csv_path)
            frame["source_file"] = csv_path.name
            frames.append(frame)
        if not frames:
            raise FileNotFoundError(f"No CSV files found in {folder}")
        frame = pd.concat(frames, ignore_index=True)
        for column in ["distance_cm", "elapsed_s"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=["distance_cm", "elapsed_s"]).copy()
        grouped = frame.groupby("source_file", sort=False)
        frame["approach_speed_cm_s"] = (grouped["distance_cm"].diff() / grouped["elapsed_s"].diff()).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        frame["label_id"] = frame["label"].astype(str).str.upper().map(LABELS).fillna(0).astype(int)
        return frame


# Single-sensor HRC Safety Log Ã¢â‚¬â€ Isolation Forest + Autoencoder

ChuyÃ¡Â»Æ’n kiÃ¡ÂºÂ¿n trÃƒÂºc IF + MLP autoencoder cÃ…Â© sang feature mÃ¡Â»â„¢t cÃ¡ÂºÂ£m biÃ¡ÂºÂ¿n. Autoencoder hÃ¡Â»Âc vÃƒÂ¹ng bÃƒÂ¬nh thÃ†Â°Ã¡Â»Âng, cÃƒÂ²n hard threshold `distance_cm` luÃƒÂ´n cÃƒÂ³ quyÃ¡Â»Ân Ã†Â°u tiÃƒÂªn cho E-Stop.

In [ ]:
from pathlib import Path
import sys, json, joblib, numpy as np, pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# load_recordings and FEATURE_COLUMNS are initialized by the Kaggle/local setup cell

In [ ]:
WARNING_CM, DANGER_CM, RANDOM_STATE = 60.0, 30.0, 42
data = load_recordings(DATA_DIR)
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer.split(data, data['label_id'], data['source_file']))
train_pool, test = data.iloc[train_idx].reset_index(drop=True), data.iloc[test_idx].reset_index(drop=True)
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE+1)
train_idx, val_idx = next(inner.split(train_pool, train_pool['label_id'], train_pool['source_file']))
train, validation = train_pool.iloc[train_idx].reset_index(drop=True), train_pool.iloc[val_idx].reset_index(drop=True)
X_train, y_train = train[FEATURE_COLUMNS], train['label_id'].astype(int)
X_val, y_val = validation[FEATURE_COLUMNS], validation['label_id'].astype(int)
X_test, y_test = test[FEATURE_COLUMNS], test['label_id'].astype(int)
classifier = ExtraTreesClassifier(n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1).fit(X_train,y_train)
normal_train = X_train[y_train == 0]
if_model = Pipeline([('scale',StandardScaler()),('detector',IsolationForest(n_estimators=500,contamination=0.05,random_state=RANDOM_STATE))]).fit(normal_train)
ae_scaler = StandardScaler().fit(normal_train)
normal_scaled = ae_scaler.transform(normal_train)
autoencoder = MLPRegressor(hidden_layer_sizes=(32,12,32), activation='relu', solver='adam', learning_rate_init=1e-3, max_iter=600, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_STATE).fit(normal_scaled, normal_scaled)

In [ ]:
def if_score(X): return -if_model.decision_function(X)
def ae_error(X):
    scaled=ae_scaler.transform(X); return np.mean((scaled-autoencoder.predict(scaled))**2,axis=1)
if_train, ae_train = if_score(normal_train), ae_error(normal_train)
if_threshold, ae_threshold = np.quantile(if_train,0.95), np.quantile(ae_train,0.95)
print('IF threshold:', if_threshold, 'AE threshold:', ae_threshold)

In [ ]:
def states(X, confidence=0.70, fusion='and'):
    pred=classifier.predict(X); conf=classifier.predict_proba(X).max(axis=1); if_flag=if_score(X)>=if_threshold; ae_flag=ae_error(X)>=ae_threshold; out=[]
    for i,distance in enumerate(X['distance_cm']):
        if distance<=DANGER_CM: out.append(('EMERGENCY',True,'HARD_DANGER_THRESHOLD')); continue
        if distance<=WARNING_CM: out.append(('WARNING',False,'HARD_WARNING_THRESHOLD')); continue
        classifier_vote=pred[i]>0 and conf[i]>=confidence; anomaly_vote=(if_flag[i] and ae_flag[i]) if fusion=='and' else (if_flag[i] or ae_flag[i]); active=classifier_vote and anomaly_vote
        out.append(('WARNING' if active else 'SAFE',False,'AI_IF_AE_'+fusion.upper() if active else 'NORMAL'))
    return pd.DataFrame(out,columns=['state','emergency_stop','reason'],index=X.index)
def score(frame,y):
    actual=(y.to_numpy()>0).astype(int); predicted=frame.state.isin(['WARNING','EMERGENCY']).astype(int).to_numpy(); return {'balanced_accuracy':float(balanced_accuracy_score(actual,predicted)),'precision':float(precision_score(actual,predicted,zero_division=0)),'recall':float(recall_score(actual,predicted,zero_division=0)),'f1':float(f1_score(actual,predicted,zero_division=0)),'confusion_matrix':confusion_matrix(actual,predicted,labels=[0,1]).tolist()}
candidates=[]
for confidence in [0.55,0.70,0.85]:
    for fusion in ['and','or']:
        m=score(states(X_val,confidence,fusion),y_val); m.update({'confidence':confidence,'fusion':fusion,'utility':2*m['recall']+m['f1']+m['balanced_accuracy']}); candidates.append(m)
best=pd.DataFrame(candidates).sort_values('utility',ascending=False).iloc[0].to_dict(); print(best)

In [ ]:
test_metrics=score(states(X_test,float(best['confidence']),str(best['fusion'])),y_test); print(json.dumps(test_metrics,indent=2))
artifact={'classifier':classifier,'isolation_forest':if_model,'autoencoder':autoencoder,'autoencoder_scaler':ae_scaler,'feature_columns':FEATURE_COLUMNS,'thresholds_cm':{'warning':WARNING_CM,'danger':DANGER_CM},'ai_confidence_threshold':float(best['confidence']),'if_anomaly_threshold':float(if_threshold),'ae_reconstruction_threshold':float(ae_threshold),'anomaly_fusion':str(best['fusion']),'architecture':'single_sensor_isolation_forest_autoencoder'}
output_dir=Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT/'ai_model'/'artifacts'; output_dir.mkdir(parents=True,exist_ok=True)
joblib.dump(artifact,output_dir/'single_sensor_if_autoencoder.joblib')
(output_dir/'single_sensor_if_autoencoder_metrics.json').write_text(json.dumps({'validation':best,'test':test_metrics},indent=2),encoding='utf-8')
print('saved:',output_dir/'single_sensor_if_autoencoder.joblib')